In [2]:
import numpy as np
import torch
from collections import deque
from sentence_transformers import SentenceTransformer, util
import transformers

transformers.logging.set_verbosity_error()

In [3]:
model = SentenceTransformer("./models/models--BAAI--bge-small-zh-v1.5/snapshots/7999e1d3359715c523056ef9478215996d62a620")

Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

In [125]:
chat_history = """
A: 中午有人去那家新开的湘菜馆吗？
B: 听说巨辣，上次我同事去吃完，下午直接请假了。
C: 哈哈，那是“工伤”吧。
D: 我带饭了，西兰花炒一切。
A: 你们不觉得带饭很麻烦吗？光洗碗就要半天。
B: 提到洗碗，现在的洗碗机器人什么时候能进化到自动收纳？
C: 那得配个人形机器人吧。
B: 对啊，现在的AI连画手都画不好，指望它收碗？
D: 扯远了，其实逻辑是一样的，都是空间识别。
A: 别空间识别了，我就想知道那家湘菜有没有团购。
C: 有个99的双人餐，但限制周末。
B: 资本的套路。
A: 哎，对了，小李你上次那个键盘修好了吗？
B: 没，轴坏了，准备换个磁轴的。
D: 磁轴最近很火，打游戏确实快。
C: 说起快，刚才快递给我打电话，我以为是诈骗。
A: 最近诈骗电话确实多。
B: 主要是数据泄露太严重。
C: 结果是我买的螺蛳粉到了，虚惊一场。
D: 螺蛳粉？你要在办公室吃吗？（惊恐）
A: 绝对不行，C你敢吃我就敢报警。
C: 开玩笑的，我回家煮。
B: 你们看今天的头条了吗？那个商业航天公司发射失败了。
D: 看到了，二级火箭没点火。
A: 航天太烧钱了，不如多研究点利民的。
C: 比如自动洗碗机？（复读机模式）
B: 啧，C你别绕回来了。
B: 我觉得航天才是人类的终极出路，资源迟早枯竭。
D: 确实，马斯克那套逻辑虽然疯狂，但挺自洽的。
A: 昨晚那场球赛谁看了？
B: 没看，在加班写需求。
C: 我看了！最后那个三分球绝了！
A: 是吧！我也觉得那球有运气成分。
D: 话说，我们下周的周报是不是要提前交？
B: 为什么？又要放假？
C: 哪来的假？清明还没到呢。
D: 老板下周出差，让我们周五下班前全发他邮箱。
A: 这种转折太硬了，我还没从球赛里走出来。
B: 别走了，生活就是一堆需求组成的。
C: 窗外好像下雨了。
A: 没下，是楼上空调滴水。
B: 这种老破小写字楼就是这样。
D: 刚才说到需求，B，你那个接口调通了吗？
B: 别提了，后端那个逻辑简直是玄学。
C: 玄学好啊，万物皆可量子力学。
A: 我妈最近就在看量子力学养生。
D: 认真的？那不是纯骗局吗？
A: 劝不动，老人家觉得那叫“高科技”。
C: 就像咱们用的那些护肤品，也一堆名词。
B: 什么玻尿酸、视黄醇？
D: 视黄醇还是有效的，是有科研数据支撑的。
B: 所以AI能预测蛋白质结构，什么时候能预测下彩票？
C: 算出来你也不敢买。
A: 哎呀，我刚才想起一件事。
D: 啥？
A: 我车位被人占了。
B: 这种事最烦人，打挪车电话。
C: 或者是那种停了很久的僵尸车？
A: 不知道，是个白色的SUV。
D: 先处理吧，别一会儿下班被堵死。
B: 继续说AI，你们觉得数字生命有意义吗？
D: 如果只是记忆备份，没有感官，那只是个复读机。
C: 但如果能模拟触觉和味觉呢？
A: 那我天天在里面吃湘菜。
B: 老王你就知道吃。
D: 这种伦理讨论最后都会变成哲学难题。
B: 比如“忒修斯之船”？
C: 那个船我知道，换了所有零件还是原来的船吗？
A: 就像我的键盘换了所有轴体，还是原来的键盘。
B: 诶，这个类比很到位。
D: 但人不一样，人有意识的连续性。
C: ……
B: ……
A: 怎么突然都不说话了？
D: 刚才老板路过我工位。
C: 吓死我了，我以为群被监控了。
B: 监控也没事，我们又没说他坏话。
A: 可是我们聊了60多条了，还没到中午。
D: 散了散了，干活。
C: 等一下！
B: 又干嘛？
C: 你们谁有充电宝？手机没电了。
A: 我有，1万毫安的。
D: 现在的手机电池技术真是瓶颈。
B: 固态电池什么时候能普及，电动车就无敌了。
C: 我还是喜欢燃油车的引擎声。
A: 算了吧，油价都涨成什么样了。
D: 刚才那个车位的事，老王你处理了吗？
A: 打电话了，对方说是走错了，马上移。
B: 这种理由真敷衍。
C: 话说，今晚有人组队打游戏吗？
D: 改天吧，今天要陪家里人吃饭。
B: 我要研究我那个坏掉的轴。
A: 我去试那家湘菜，有人去吗？
C: 走起，我带个解辣神器。
B: 算我一个，大不了下午请假。
D: 你们真是……记得发朋友圈。
康
A: OK，11:50 楼下集合。
B: 别迟到，老王。
C: 准时出发！
"""
history_list = chat_history.strip().split("\n")

In [126]:
def sentence_weight(text: str):
    cn = jp = kr = 0
    others: list[str] = []
    for char in text:
        code = ord(char)
        if 0x4E00 <= code <= 0x9FFF:
            cn += 1
        elif 0x3040 <= code <= 0x309F or 0x30A0 <= code <= 0x30FF:
            jp += 1
        elif 0xAC00 <= code <= 0xD7AF or 0x1100 <= code <= 0x11FF:
            kr += 1
        else:
            others.append(char)
    return (cn / 2) + (jp / 4) + (kr / 5) + len("".join(others).split())

In [187]:
class TopicDecoupler:
    def __init__(self, model: SentenceTransformer, alpha: float = 0.8):
        """
        话题解藕器，用于判断输入的句子是否属于同一话题。
        Args:
            model (SentenceTransformer): SentenceTransformer 模型。
            alpha (float): 指数衰减权重。
        """
        emb_dim = model.get_sentence_embedding_dimension()
        if emb_dim is None:
            raise ValueError("SentenceTransformer model does not have a fixed sentence embedding dimension.")
        self.model = model
        self.context_emb = torch.zeros(emb_dim, device=self.model.device)
        self.scores_history: list[float] = []
        self.weights_history: list[float] = []
        self.alpha = alpha
        self.topic_change = self._topic_change_0

    def _topic_change_0(self, sentence_emb: torch.Tensor, weight: float):
        self.weights_history.append(weight)
        self.topic_change = self._topic_change_1
        return False

    def _topic_change_1(self, sentence_emb: torch.Tensor, weight: float):
        self.weights_history.append(weight)
        self.scores_history.append(util.cos_sim(sentence_emb, self.context_emb).item())
        self.topic_change = self._topic_change_2
        return False

    def _topic_change_2(self, sentence_emb: torch.Tensor, weight: float):
        score = util.cos_sim(sentence_emb, self.context_emb).item()
        scores_history = np.array(self.scores_history)
        threshold = scores_history.mean() - max(scores_history.std(), 0.01)
        if flag := (score < threshold):
            self.weights_history.clear()
            self.scores_history.clear()
            self.topic_change = self._topic_change_1
        elif scores_history.size > 3:
            self.topic_change = self._topic_change_3
        self.weights_history.append(weight)
        self.scores_history.append(score)
        return flag

    def _topic_change_3(self, sentence_emb: torch.Tensor, weight: float):
        score = util.cos_sim(sentence_emb, self.context_emb).item()
        scores_history = np.array(self.scores_history)
        weights_history = np.array(self.weights_history)
        Q1, Q2, Q3 = np.quantile(scores_history, [0.25, 0.5, 0.75])
        scale = (weight - weights_history.mean()) * scores_history.std() / max(weights_history.std(), 0.01)
        threshold = Q1 + Q2 - Q3 + scale
        if flag := (score < threshold):
            self.weights_history.clear()
            self.scores_history.clear()
            self.topic_change = self._topic_change_1
        self.weights_history.append(weight)
        self.scores_history.append(score)
        return flag

    @torch.no_grad()
    def step(self, sentence: str):
        if not sentence:
            return False
        sentence_emb = self.model.encode(sentence, convert_to_tensor=True)
        weight = sentence_weight(sentence)
        if self.topic_change(sentence_emb, weight):
            self.context_emb.mul_(0.5).add_(sentence_emb, alpha=0.5)
            return True
        else:
            # self.context_emb.add_((sentence_emb - self.context_emb) / self.__step_count)
            self.context_emb.mul_(self.alpha).add_(sentence_emb, alpha=1 - self.alpha)
            return False

In [188]:
decoupler = TopicDecoupler(model)
for line in history_list:
    print(decoupler.step(line), f"{line[:30]}..." if len(line) > 30 else line)

False A: 中午有人去那家新开的湘菜馆吗？
False B: 听说巨辣，上次我同事去吃完，下午直接请假了。
True C: 哈哈，那是“工伤”吧。
False D: 我带饭了，西兰花炒一切。
False A: 你们不觉得带饭很麻烦吗？光洗碗就要半天。
True B: 提到洗碗，现在的洗碗机器人什么时候能进化到自动收纳？
False C: 那得配个人形机器人吧。
False B: 对啊，现在的AI连画手都画不好，指望它收碗？
False D: 扯远了，其实逻辑是一样的，都是空间识别。
False A: 别空间识别了，我就想知道那家湘菜有没有团购。
False C: 有个99的双人餐，但限制周末。
False B: 资本的套路。
True A: 哎，对了，小李你上次那个键盘修好了吗？
False B: 没，轴坏了，准备换个磁轴的。
False D: 磁轴最近很火，打游戏确实快。
False C: 说起快，刚才快递给我打电话，我以为是诈骗。
False A: 最近诈骗电话确实多。
False B: 主要是数据泄露太严重。
False C: 结果是我买的螺蛳粉到了，虚惊一场。
False D: 螺蛳粉？你要在办公室吃吗？（惊恐）
False A: 绝对不行，C你敢吃我就敢报警。
False C: 开玩笑的，我回家煮。
True B: 你们看今天的头条了吗？那个商业航天公司发射失败了。
False D: 看到了，二级火箭没点火。
False A: 航天太烧钱了，不如多研究点利民的。
True C: 比如自动洗碗机？（复读机模式）
False B: 啧，C你别绕回来了。
False B: 我觉得航天才是人类的终极出路，资源迟早枯竭。
False D: 确实，马斯克那套逻辑虽然疯狂，但挺自洽的。
False A: 昨晚那场球赛谁看了？
False B: 没看，在加班写需求。
False C: 我看了！最后那个三分球绝了！
False A: 是吧！我也觉得那球有运气成分。
True D: 话说，我们下周的周报是不是要提前交？
False B: 为什么？又要放假？
False C: 哪来的假？清明还没到呢。
False D: 老板下周出差，让我们周五下班前全发他邮箱。
True A: 这种转折太硬了，我还没从球赛里走出来。
False B: 别走了，生活就是一堆需求组成的。
F